In [ ]:
from pathlib import Path
import pandas as pd
import optuna

import sys
sys.path.append("../src")

from crop_model_calibration_and_run import *

## Calibrated crop parameters

### MM

In [ ]:
best_params_mm_ythr1pct_left_0p6_var = {
    'Zmax': 1.6557119653458805,
    'WP': 18.23099479827126,
    'HI0': 0.1933251119254694,
    'CGC': 0.006132622179024764
}

### TH

In [ ]:
best_params_th_ythr1pct_left_0p6_var = {
    'HI0': 0.3484648524767283,
    'WP': 31.21342736382409,
    'CGC': 0.006776932907851044,
    'CCx': 0.38502353986594234
}

### KH

In [ ]:
best_params_kh_ythr1pct_left_0p6_var = {
    'WP': 15.693427677463774,
    'HI0': 0.19247259070768555,
    'CGC': 0.005605491874365324,
    'CCx': 0.53051594245777
}

### VN

In [ ]:
best_params_vn_ythr1pct_left_0p6_var = {
    'WP': 16.63469690974607,
    'CGC': 0.00616699732224831,
    'HI0': 0.20872808030444145,
    'CCx': 0.8829174599663626
}

### PH

In [ ]:
best_params_ph_ythr1pct_left_0p6_var = {
    'CGC': 0.005506831809368263,
    'WP': 13.695439489034968,
    'HI0': 0.31966314584073774,
    'CCx': 0.6928728859271447
}

### ID

In [ ]:
best_params_id_ythr1pct_left_0p5_right_0p1_var = {
    'WP': 19.36797361389945,
    'HI0': 0.25139644492363367,
    'CGC': 0.006747183306158826,
    'CCx': 0.7895575677061318
}

## Run
One country/case at one time
### KH/MM/PH/TH/VN (plant and harvest in the same calendar year)

In [ ]:
dict_planting_data = {
    "ID": "10/01",
    "KH": "07/01",
    "MM": "05/01",
    "PH": "04/01",
    "TH": "05/01",
    "VN": "01/01"
}

dict_best_parameters = {
    "ID": best_params_id_ythr1pct_left_0p5_right_0p1_var,
    "KH": best_params_kh_ythr1pct_left_0p6_var,
    "MM": best_params_mm_ythr1pct_left_0p6_var,
    "PH": best_params_ph_ythr1pct_left_0p6_var,
    "TH": best_params_th_ythr1pct_left_0p6_var,
    "VN": best_params_vn_ythr1pct_left_0p6_var
}

In [ ]:
country_code = "MM"

gid_harvest_area_df = pd.read_csv(
    f"../../Results/1a_CAs/HAs_{country_code}.csv"
)
gid_harvest_area_df = gid_harvest_area_df[["GID", "maize"]]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    f"../../Results/3.1_DetrendedYield/DetrendedYield_{country_code}_maize.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / country_code
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

simulation_results_df, comparison_df = simulate_country_yield(
    gid_harvest_area_valid_df,
    weather_dfs,
    observed_yield_df,
    best_crop_parameters=dict_best_parameters[country_code],
    crop_name="MaizeGDD",
    planting_date=dict_planting_data[country_code],
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=0.01*observed_yield_df["yield_obs_t_ha"].mean(),
)

In [ ]:
simulation_results_df["growing_days"] = (
    pd.to_datetime(simulation_results_df["harvest_date"], format="%Y-%m-%d")
    - pd.to_datetime(
        simulation_results_df["harvest_date"].dt.year.astype(str)
            .add("/")
            .add(simulation_results_df["planting_date"]),
        format="%Y/%m/%d"
    )
).dt.days

print(f"Min growing days: {simulation_results_df["growing_days"].min()}")
print(f"Max growing days: {simulation_results_df["growing_days"].max()}")

simulation_results_df.head()

### ID (one growing season spanning two calendar years, harvest in the 2nd calendar year)

In [ ]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_ID.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'maize']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_ID_maize.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "ID"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

simulation_results_df, comparison_df = simulate_country_yield(
    gid_harvest_area_valid_df,
    weather_dfs,
    observed_yield_df,
    best_crop_parameters=best_params_id_ythr1pct_left_0p5_right_0p1_var,
    crop_name="MaizeGDD",
    planting_date="10/01",
    user_harvest_year=np.arange(1992, 2021),
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=0.01*observed_yield_df["yield_obs_t_ha"].mean(),
)

In [ ]:
simulation_results_df["growing_days"] = (
    pd.to_datetime(simulation_results_df["harvest_date"], format="%Y-%m-%d")
    - pd.to_datetime(
        (simulation_results_df["year"]-1).astype(str)
            .add("/")
            .add(simulation_results_df["planting_date"]),
        format="%Y/%m/%d"
    )
).dt.days

print(f"Min growing days: {simulation_results_df["growing_days"].min()}")
print(f"Max growing days: {simulation_results_df["growing_days"].max()}")

simulation_results_df.head()

## Check and save files

In [ ]:
grid_yield_output_path = Path(
    "../../Results/3.2_SimulatedYield/"
    "aquacrop_maize_grid_yield_mm_ythr1pct_left_0p6_var_260717.csv"
)

country_ensemble_yield_output_path = Path(
    "../../Results/3.2_SimulatedYield/"
    "aquacrop_maize_country_ensemble_yield_"
    "mm_ythr1pct_left_0p6_var_260717.csv"
)

In [ ]:
comparison_df.head()

In [ ]:
simulation_results_df = simulation_results_df.round(
    {
        "yield_sim_t_ha": 4,
        "harvest_area_ha": 2,
    }
)

comparison_df = comparison_df.round(
    {
        "yield_sim_t_ha": 2,
        "yield_obs_t_ha": 2,
    }
)

simulation_results_df.to_csv(grid_yield_output_path, index=False)
comparison_df.to_csv(country_ensemble_yield_output_path, index=False)